# MNIST Classification: Continual Learning with Hamiltonian Gradients

This notebook demonstrates continual learning on MNIST classification with rotation/scaling transforms.

## Overview
- **Dataset**: MNIST digits with rotation and scaling transforms across tasks
- **Model**: Convolutional Neural Network (CNN)
- **Methods**: Baseline vs AWB Full (Adaptive Weight Basis)
- **Metrics**: Accuracy, Backward Transfer, Forgetting

## Contents
1. [Part 1: Quick Training Demo](#part1) - Run minimal training (requires compute)
2. [Part 2: Load Pre-computed Results](#part2) - Analyze saved experiments
3. [Part 3: Comparison Plot](#part3) - Compare Baseline vs AWB

In [ ]:
# Setup
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)
sys.path.insert(0, os.path.join(project_root, 'src'))
sys.path.insert(0, os.path.join(project_root, 'examples'))

import numpy as np
import matplotlib.pyplot as plt
from utils import load_results, compute_cl_metrics, plot_comparison, print_experiment_summary

print(f"Project root: {project_root}")

<a id='part1'></a>
## Part 1: Quick Training Demo (Optional)

This section demonstrates how to run a minimal training experiment. **Skip this if you don't have JAX/GPU available.**

In [ ]:
# Optional: Run minimal training demo
# Uncomment the lines below to run

# import jax
# from cl.config import load_config
# from cl.runners import train_model
# 
# # Check JAX backend
# print(f"JAX Backend: {jax.default_backend()}")
# print(f"JAX Devices: {jax.devices()}")
# 
# # Load example config (debug mode for quick demo)
# config_path = os.path.join(project_root, 'examples/configs/mnist_baseline.json')
# config = load_config(config_path)
# 
# print(f"\nConfig loaded:")
# print(f"  Dataset: {config.get('data')}")
# print(f"  Network: {config.get('network')}")
# print(f"  Tasks: {config.get('n_task')}")
# print(f"  Epochs per task: {config.get('epochs_per_task')}")
# print(f"  Debug mode: {config.get('debug_mode')}")
# 
# # Run training (takes ~2-3 minutes with debug mode)
# record_dict = train_model(config, run_id=0)

<a id='part2'></a>
## Part 2: Load Pre-computed Results

Load and explore the structure of saved experiment results.

In [ ]:
# Load pre-computed results
data_dir = os.path.join(project_root, 'examples/data/mnist')

baseline_data = load_results(os.path.join(data_dir, 'baseline_run0.pkl'))
awb_data = load_results(os.path.join(data_dir, 'awb_full_run0.pkl'))

print("Loaded baseline and AWB results!")

In [ ]:
# Explore data structure
print("Keys in data dictionary:")
print(list(baseline_data.keys()))

print("\nMetadata:")
for key, value in baseline_data.get('metadata', {}).items():
    print(f"  {key}: {value}")

In [ ]:
# Print experiment summaries
print_experiment_summary(baseline_data, name='MNIST Baseline')
print_experiment_summary(awb_data, name='MNIST AWB Full')

### MNIST Task Structure

Each task applies different transformations to MNIST:
- **Rotation**: Random rotation angle
- **Scaling**: Random scaling factor

This creates distribution shift between tasks, testing the model's ability to:
1. Learn new transformations without forgetting old ones
2. Transfer knowledge across related tasks

In [ ]:
# Look at task performance matrix
perf_matrix = baseline_data.get('task_performance_matrix', {})
n_tasks = len(perf_matrix)

print(f"Number of tasks: {n_tasks}")
print("\nPerformance matrix (rows=evaluation time, cols=task ID):")

# Convert to numpy for visualization
matrix = np.zeros((n_tasks, n_tasks))
for j in range(n_tasks):
    for i in range(n_tasks):
        if str(i) in perf_matrix.get(j, {}):
            matrix[j, i] = perf_matrix[j][str(i)]

# Show as heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix, cmap='viridis', vmin=0, vmax=1)
ax.set_xlabel('Task ID')
ax.set_ylabel('Evaluation After Task')
ax.set_title('MNIST Baseline - Task Performance Matrix')
plt.colorbar(im, label='Accuracy')
plt.show()

<a id='part3'></a>
## Part 3: Comparison Plot

Create a publication-quality 4-panel comparison of Baseline vs AWB methods.

In [ ]:
# Create comparison plot
fig = plot_comparison(
    baseline_data, 
    awb_data, 
    metric_type='accuracy',
    title_prefix='MNIST Classification'
)
plt.show()

In [ ]:
# Optional: Save figure
# fig.savefig('mnist_comparison.pdf', dpi=300, bbox_inches='tight')

## Summary

### Key Observations

1. **Test Accuracy**: AWB maintains higher accuracy across tasks
2. **Hamiltonian Loss**: AWB shows more consistent optimization
3. **Gradient Norm**: AWB often shows smoother gradient dynamics
4. **CL Metrics**:
   - AWB achieves better average accuracy
   - AWB shows better backward transfer

### MNIST Challenges for Continual Learning

- **Rotation transforms**: Model must learn rotation-invariant features
- **Scaling transforms**: Model must handle different image scales
- **Combined effect**: Creates significant distribution shift between tasks

### How AWB Helps

AWB addresses these challenges through:
1. **Architecture adaptation**: CNN layers can grow to handle new transforms
2. **A/B transfer matrices**: Smooth knowledge transfer preserves old features
3. **Task warmup**: Gradual LR increase stabilizes learning